In [12]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Cerebrasensecloud')
OASIS2_BUNDLE_ROOT = DRIVE_ROOT / 'OASIS-2'
RUNTIME_ROOT = DRIVE_ROOT / 'backend_runtime'
RUN_NAME = 'oasis2_colab_baseline_v2'

for name, path in {
    'DRIVE_ROOT': DRIVE_ROOT,
    'OASIS2_BUNDLE_ROOT_requested': OASIS2_BUNDLE_ROOT,
    'RUNTIME_ROOT': RUNTIME_ROOT,
}.items():
    print(name, path.exists(), path)

print('Note: the runner can now auto-detect common OASIS-2 upload layouts even if the exact OASIS-2 folder name/path differs.', flush=True)




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_ROOT True /content/drive/MyDrive/Cerebrasensecloud
OASIS2_BUNDLE_ROOT_requested True /content/drive/MyDrive/Cerebrasensecloud/OASIS-2
RUNTIME_ROOT True /content/drive/MyDrive/Cerebrasensecloud/backend_runtime
Note: the runner can now auto-detect common OASIS-2 upload layouts even if the exact OASIS-2 folder name/path differs.


In [13]:
import shutil
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path('/content/cerebrasense')
BACKEND_ROOT = REPO_ROOT / 'alz_backend'
REPO_URL = 'https://github.com/Billrichard209/cerebrasense.git'
REQUIRED_COMMIT = 'e577752'

def run_checked(cmd, *, cwd=None, label=None):
    print(f'RUNNING {label or cmd[0]}: {' '.join(cmd)}', flush=True)
    completed = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr, flush=True)
        raise RuntimeError(f"Command failed ({label or cmd[0]}): {' '.join(cmd)}")
    if completed.stderr:
        print(completed.stderr, flush=True)
    return completed

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

run_checked(['git', 'clone', REPO_URL], cwd='/content', label='git-clone')

if not BACKEND_ROOT.exists():
    raise FileNotFoundError(f'Expected backend after clone: {BACKEND_ROOT}')

requirements = BACKEND_ROOT / 'requirements-colab.txt'
if not requirements.exists():
    raise FileNotFoundError(f'Missing requirements file: {requirements}')

run_checked(
    [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', str(requirements)],
    cwd=str(BACKEND_ROOT),
    label='pip-install',
)

repo_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
run_checked(['git', 'merge-base', '--is-ancestor', REQUIRED_COMMIT, 'HEAD'], cwd=str(REPO_ROOT), label='git-merge-base')
print('repo_root =', REPO_ROOT)
print('backend_root =', BACKEND_ROOT)
print('repo_commit =', repo_commit)
print('required_commit =', REQUIRED_COMMIT)



RUNNING git-clone: git clone https://github.com/Billrichard209/Cerebrasense-.git
Cloning into 'Cerebrasense-'...

RUNNING pip-install: /usr/bin/python3 -m pip install --disable-pip-version-check -r /content/Cerebrasense-/alz_backend/requirements-colab.txt

RUNNING git-merge-base: git merge-base --is-ancestor a6fc5f9 HEAD
repo_root = /content/Cerebrasense-
backend_root = /content/Cerebrasense-/alz_backend
repo_commit = e9a6aeb37cffb3146cfd920110ff4851ea0f4d23
required_commit = a6fc5f9


In [16]:
#check
from pathlib import Path
import pandas as pd

run_root = Path('/content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_baseline')
print('run_root:', run_root.exists(), run_root)

metrics = run_root / 'metrics' / 'epoch_metrics.csv'
best_ckpt = run_root / 'checkpoints' / 'best_model.pt'
last_ckpt = run_root / 'checkpoints' / 'last_model.pt'

for p in [metrics, best_ckpt, last_ckpt]:
    print(p.name, p.exists())

if metrics.exists():
    df = pd.read_csv(metrics)
    print(df.tail())


run_root: True /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_baseline
epoch_metrics.csv True
best_model.pt True
last_model.pt True
   epoch  learning_rate  train_loss  val_loss  accuracy     auroc  precision  \
2      3         0.0001    0.681520  1.213856  0.500000  0.368999   0.500000   
3      4         0.0001    0.700501  0.748788  0.462963  0.493827   0.454545   
4      5         0.0001    0.667674  0.956465  0.444444  0.436214   0.451613   
5      6         0.0001    0.652128  0.717984  0.555556  0.567215   0.636364   
6      7         0.0001    0.640216  0.753425  0.481481  0.500000   0.482759   

     recall        f1  sensitivity  specificity  train_batches  val_batches  
2  1.000000  0.666667     1.000000     0.000000            259           54  
3  0.370370  0.408163     0.370370     0.555556            259           54  
4  0.518519  0.482759     0.518519     0.370370            259           54  
5  0.259259  0.368421     0.2592

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    'scripts/train_oasis2_colab.py',
    '--project-root', str(BACKEND_ROOT),
    '--runtime-root', str(RUNTIME_ROOT),
    '--bundle-root', str(OASIS2_BUNDLE_ROOT),
    '--run-name', RUN_NAME,
    '--epochs', '20',
    '--batch-size', '1',
    '--gradient-accumulation-steps', '1',
    '--num-workers', '0',
    '--image-size', '64', '64', '64',
    '--seed', '42',
    '--split-seed', '42',
    '--device', 'auto',
]

print('RUNNING:', ' '.join(cmd), flush=True)
process = subprocess.Popen(
    cmd,
    cwd=BACKEND_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'OASIS-2 training runner failed with exit code {return_code}')

print('OASIS-2 training runner completed successfully.', flush=True)



RUNNING: /usr/bin/python3 scripts/train_oasis2_colab.py --project-root /content/Cerebrasense-/alz_backend --runtime-root /content/drive/MyDrive/Cerebrasensecloud/backend_runtime --bundle-root /content/drive/MyDrive/Cerebrasensecloud/OASIS-2 --run-name oasis2_colab_baseline --epochs 20 --batch-size 1 --gradient-accumulation-steps 1 --num-workers 0 --image-size 64 64 64 --seed 42 --split-seed 42 --device auto
Starting OASIS-2 Colab bundle pipeline
Resolved OASIS-2 bundle root: /content/drive/MyDrive/Cerebrasensecloud/OASIS-2
Runtime data root: /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/data
Runtime outputs root: /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs
Inspecting uploaded OASIS-2 bundle
Upload bundle inspection status: pass
Resolving OASIS-2 metadata template source
Resolved metadata template source: /content/drive/MyDrive/Cerebrasensecloud/OASIS-2/backend_reference/oasis2_metadata_template.csv
Running preflight runtime refresh from Drive bundle
Refr

In [11]:
from pathlib import Path
import json

blocked_summary_path = RUNTIME_ROOT / 'outputs' / 'reports' / 'onboarding' / 'oasis2_colab_bundle_summary.json'
trained_summary_path = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME / 'reports' / 'colab_run_summary.json'

print('blocked_summary_path =', blocked_summary_path)
print('trained_summary_path =', trained_summary_path)

summary_candidates = [trained_summary_path, blocked_summary_path]
existing_summaries = [path for path in summary_candidates if path.exists()]
if not existing_summaries:
    raise FileNotFoundError('Expected an OASIS-2 summary JSON, but none was found in the runtime outputs.')

summary_path = max(existing_summaries, key=lambda path: path.stat().st_mtime)
print('using_summary_path =', summary_path)

summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2))
print('training_ready =', summary.get('training_ready'))
print('training_started =', summary.get('training_started'))
print('blocked_reason =', summary.get('blocked_reason'))
print('run_root =', summary.get('run_root'))
print('best_checkpoint =', summary.get('best_checkpoint'))


blocked_summary_path = /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/reports/onboarding/oasis2_colab_bundle_summary.json
trained_summary_path = /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_baseline/reports/colab_run_summary.json
using_summary_path = /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_baseline/reports/colab_run_summary.json
{
  "generated_at": "2026-05-02T11:06:32.418068+00:00",
  "bundle_root": "/content/drive/MyDrive/Cerebrasensecloud/OASIS-2",
  "bundle_source_for_checks": "/content/drive/MyDrive/Cerebrasensecloud/OASIS-2",
  "source_root_for_training": "/content/drive/MyDrive/Cerebrasensecloud/OASIS-2",
  "runtime_data_root": "/content/drive/MyDrive/Cerebrasensecloud/backend_runtime/data",
  "runtime_outputs_root": "/content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs",
  "metadata_template_source_path": "/content/drive/MyDrive/Cerebrasensecloud/OASIS-2/